This notebook outlines the process of data loading and provides a brief description of the *DocTamper* dataset, which contains images of documents modified for tampering purposes. For each image, there is a corresponding annotation that specifies the areas where the documents were altered.

In [ ]:
!pip3 install lmdb

In [ ]:
import lmdb
import numpy as np
from PIL import Image
from io import BytesIO
from pathlib import Path
from random import sample
import matplotlib.pyplot as plt
data_path = Path(".")

# Extract data

The files in the dataset can be read using the `lmdb` library. The following cell defines an object that provides access to one of the files in the dataset.

In [5]:
env = lmdb.open(
    str(data_path/"DocTamperV1-SCD"),
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False
)

The dataset comprises pairs of files:

- `image-<index>`: the original document image.
- `label-<index>`: an image highlighting the modified pixels.

In [ ]:
with env.begin(write=False) as txn:
    image_buffer = txn.get(('image-000000001').encode('utf-8'))
    label_buffer = txn.get(('label-000000001').encode('utf-8'))

image_buffer[:100]

When you load an image, it is initially a set of bytes. The following cell demonstrates how to transform such an image into a `PIL.Image.Image` object, which can then be displayed.

In [ ]:
image = Image.open(BytesIO(image_buffer)).convert("RGBA")
image

Same for markdown.

In [ ]:
label = Image.open(BytesIO(label_buffer))
label

The following snippet shows how to blend the mask with the original image, making it easier to understand the modified areas.

In [ ]:
mask = np.array(label)

tmp = np.tile([0, 0, 0, 0], (*mask.shape, 1)).astype(np.uint8)
tmp[mask == 255] = [255, 0, 0, 255]
mask = Image.fromarray(tmp)
del tmp

Image.blend(im1=image, im2=mask, alpha=0.3)

## Overview

The dataset contains several files, each serving a distinct purpose. In this section, we'll examine them.

The following cell creates an environment for each database provided by the dataset authors.

In [ ]:
environments = {
    data_group.parts[-1] : lmdb.open(
        str(data_group),
        readonly=True,
        lock=False,
        readahead=False,
        meminit=False
    )
    for data_group in data_path.iterdir()
}
environments

Some tools that display a given set of images.

In [37]:
def get_masked_image(env: lmdb.Environment, ind: int) -> Image.Image:
    '''
    Read image and label from the given environment and represent it as an image 
    where the mask is highlighted.
    Parameters
    ----------
    env : lmdb.Environment
        Environment for searching the file.
    ind : int
        Index of the image.
        
    Returns
    -------
    out : Image.Image
        Resulting image with highlighted mask.
    '''
    with env.begin(write=False) as txn:
        image_buffer = txn.get(('image-%09d' % ind).encode('utf-8'))
        label_buffer = txn.get(('label-%09d' % ind).encode('utf-8'))
    
    image = Image.open(BytesIO(image_buffer)).convert("RGBA")
    label = Image.open(BytesIO(label_buffer))
    
    mask = np.array(label)

    tmp = np.tile([0, 0, 0, 0], (*mask.shape, 1)).astype(np.uint8)
    tmp[mask == 255] = [255, 0, 0, 255]
    mask = Image.fromarray(tmp)

    return Image.blend(im1=image, im2=mask, alpha=0.3)

def display_images_in_row(images: list[Image.Image]) -> None:
    '''
    Display given list of pillow images in a row.
    
    Parameters
    ----------
    images: list[Image.Image]
        List of images that have to be displayed.
    '''
    fig, axes = plt.subplots(1, len(images), figsize=(len(images) * 4, 6))
    
    for ax, img in zip(axes, images):
        ax.imshow(img)
        ax.axis('off')

    plt.show()

**DocTamperV1-SCD**: We have previously examined some examples from this subset, which appears to consist of modified receipts.

In [ ]:
display_images_in_row([
    get_masked_image(env=environments['DocTamperV1-SCD'], ind=i) 
    for i in sample(population=range(1,200), k=5)
])

**DocTamperV1-FCD**: This subset appears to contain photos of book pages with modified words. The modifications are evident, as the font significantly differs from the original.

In [ ]:
display_images_in_row([
    get_masked_image(env=environments['DocTamperV1-FCD'], ind=i) 
    for i in sample(population=range(1,200), k=5)
])

**DocTamperV1-TrainingSet** and **DocTamperV1-TestingSet** appear similar, as they seem to be a mix of the previously shown subsets. Based on their names, it is likely that the former is used for model training, while the latter serves as the validation dataset.

In [ ]:
display_images_in_row([
    get_masked_image(env=environments['DocTamperV1-TrainingSet'], ind=i) 
    for i in sample(population=range(1,200), k=5)
])
display_images_in_row([
    get_masked_image(env=environments['DocTamperV1-TestingSet'], ind=i) 
    for i in sample(population=range(1,200), k=5)
])